In [ ]:
!nvidia-smi | head -12

# GPU benchmark: TokenVector.Inference vs PyTorch trên Colab GPU

So sánh train GPT-2 124M forward+backward trên:
- **.tkv** (CPU AVX2 + CUDA-native dispatch — nếu GPU khả dụng)
- **PyTorch** (CUDA)

Workload: cùng fixture 124M (sinh từ `gen_gpt2_fixture.py --full`), seq1024,
đo bằng `sem_bench_124m_seq.tkv` (mode 0 = fwd+bwd).

> Chú ý: nếu GPU không có (Colab CPU), .tkv sẽ tự fallback sang AVX2/CPU.
> Đọc kết quả theo: `.tkv fwd+bwd` và `torch fwd+bwd`.


In [ ]:
# [1/5] Install .NET 10 + build-essential
!apt-get update -qq && apt-get install -y -qq build-essential gcc >/dev/null
!curl -fsSL https://dot.net/v1/dotnet-install.sh -o /tmp/dotnet-install.sh
!bash /tmp/dotnet-install.sh --channel 10.0 --runtime dotnet --install-dir /opt/dotnet
import os
os.environ["PATH"] = "/opt/dotnet:" + os.environ["PATH"]
!/opt/dotnet/dotnet --version


In [ ]:
# [2/5] Clone repos + cài CUDA(runtime) package (để dlopen libcublas)
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.git
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git
!pip install -q nvidia-cublas-cu12 nvidia-cuda-nvcc-cu12 2>&1 | tail -2
import sys, glob, os
for p in sys.path:
    for name in ('cublas', 'cuda-runtime'):
        so = f'{p}/nvidia/{name}/lib/libcublas.so.12' if name == 'cublas' else None
        if so and os.path.exists(so): print('Cublas found:', so)


In [ ]:
# [3/5] Build native library .so với CUDA path (đt 1) + CPU fallback (AVX2)
!cd TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC \
  tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c \
  -lm -lpthread -ldl -o tkv_simd_native.dll
!cp TokenVector/Release/tools/simd_native/tkv_simd_native.dll TokenVector.Inference/
print('Built .so — CPU fallback confirmed, CUDA path ready for GPU')


In [ ]:
# [4/5] Sinh fixture 124M (một lần, ~2.3 GB)
!python3 TokenVector.Inference/tools/gen_gpt2_fixture.py --full --out /content/fixture-124m 2>&1 | tail -4
!echo /content/fixture-124m > TokenVector.Inference/tv/tests/g2_full_dir.txt
!head -1 TokenVector.Inference/tv/tests/g2_full_dir.txt


In [ ]:
# [5/5] Bench .tkv vs PyTorch trên GPU (seq1024, fwd+bwd)
!cd TokenVector.Inference && printf '1024\n0\n1\n' > tv/tests/_seq_cfg.txt
!cd TokenVector.Inference && /opt/dotnet/dotnet demo/bseq.dll 2>&1 | tail -4
print('--- PyTorch reference ---')
!cd TokenVector.Inference && python3 -c "import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a')"
!cd TokenVector.Inference && python3 benchmark/bench_torch.py d 1024 2>&1 | tail -3
